# Approfondimento 1 · I tre approcci ai dati mancanti

**CAF GIAS, Università degli Studi Link, modulo 2.4** · Michele Ferramola · slide «Togliere i dati mancanti», «Interpolazione del dato mancante», «Dato mancante come predittore?»

**Il messaggio.** Nessun approccio è giusto in assoluto: ognuno regge sotto un'ipotesi precisa su perché il dato manca.

**I dati.** NHANES 2013-2014, 6.113 adulti. Per sapere se un approccio regge bisogna conoscere il valore vero:
prendiamo la pressione sistolica, che quasi tutti hanno, e la togliamo noi, in tre modi diversi.

### Approcci 1 e 2 · Tenere solo chi ha il dato, oppure stimare il dato mancante

- **MCAR:** la pressione manca per caso, al 40% dei pazienti.
- **MAR:** manca al 70% dei giovani (sotto i 40 anni) e al 25% degli altri. L'età è registrata.
- **MNAR:** manca al 70% di chi ha la pressione alta (130 o più) e al 25% degli altri.

Per ciascun modo: di quanto sbaglia la media chi tiene solo i pazienti con il dato, e chi stima il dato mancante da età e sesso.

**Atteso:** MCAR +0,1 e +0,1 · MAR +2,6 e −0,2 · MNAR −4,8 e −4,0 mmHg.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import roc_auc_score

DATI = "https://raw.githubusercontent.com/MicheleFerramola00/lezioni/caf-unilink-m2.4/dati/"
V = pd.read_parquet(DATI + "valori_nhanes_adulti.parquet")
p = V.dropna(subset=["pas"])
vero = p.pas.mean()
print(f"adulti con la pressione misurata: {len(p)}, pressione sistolica media vera: {vero:.1f} mmHg")

rng = np.random.default_rng(0)
regole = {"MCAR, manca per caso": np.full(len(p), 0.40),
          "MAR, manca ai giovani": np.where(p.eta < 40, 0.70, 0.25),
          "MNAR, manca a chi ha la pressione alta": np.where(p.pas >= 130, 0.70, 0.25)}
righe = []
for nome, probabilita in regole.items():
    manca = rng.random(len(p)) < probabilita
    stima = LinearRegression().fit(p.loc[~manca, ["eta", "donna"]], p.pas[~manca]).predict(p[["eta", "donna"]])
    righe.append((nome, p.pas[~manca].mean() - vero, np.where(manca, stima, p.pas).mean() - vero))
pd.DataFrame(righe, columns=["come manca", "errore tenendo solo chi ha il dato (mmHg)",
                             "errore stimando il dato da età e sesso (mmHg)"]).round(1)

- **Tenere solo chi ha il dato** regge se il dato manca per caso. Se manca ai giovani, la media sale.
- **Stimare il dato mancante** corregge quando il motivo della mancanza è registrato: qui l'età.
- **Se manca proprio a chi ha la pressione alta** sbagliano tutti e due: nessun calcolo recupera il dato che manca.

### Approccio 3 · Il dato mancante come predittore

Predittore è una delle variabili da cui il modello calcola la previsione. Qui la variabile è il fatto stesso che l'esame manchi.

Due ospedali con pazienti simili. Nell'ospedale **A** la glicemia si chiede a chi è sospetto di diabete;
nell'ospedale **B** si chiede a tutti sopra i 50 anni. Un modello impara in A a riconoscere il diabete, poi viene usato in B.

**Atteso:** chi usa la mancanza passa da 0,906 in A a 0,816 in B; chi stima il valore mancante da 0,849 a 0,858.

In [ ]:
q = V.dropna(subset=["glicemia", "bmi"]).copy()
q["diabete"] = ((q.diabete_dichiarato == 1) | (q.hba1c >= 6.5)).astype(int)
rng = np.random.default_rng(0)
in_A = rng.random(len(q)) < 0.5
A, B = q[in_A].copy(), q[~in_A].copy()
A["misurata"] = rng.random(len(A)) < np.where(A.diabete == 1, 0.85, 0.30)   # A: si chiede a chi è sospetto
B["misurata"] = rng.random(len(B)) < np.where(B.eta >= 50, 0.90, 0.10)      # B: si chiede a tutti sopra i 50 anni
impara = rng.random(len(A)) < 0.7
A_imp, A_ver = A[impara], A[~impara]

mediana = A_imp.glicemia[A_imp.misurata].median()
stima = LinearRegression().fit(A_imp.loc[A_imp.misurata, ["eta", "bmi"]], A_imp.glicemia[A_imp.misurata])

def colonne(x, usa_la_mancanza):
    g = x.glicemia.where(x.misurata)
    X = pd.DataFrame({"eta": x.eta, "bmi": x.bmi})
    if usa_la_mancanza:     # approccio 3: una colonna dice se il dato manca
        X["glicemia"], X["manca"] = g.fillna(mediana), (~x.misurata).astype(int)
    else:                   # approccio 2: il dato mancante si stima da età e BMI
        X["glicemia"] = g.fillna(pd.Series(stima.predict(x[["eta", "bmi"]]), index=x.index))
    return X

righe = []
for nome, usa in (("usa la mancanza come predittore", True), ("stima il valore che manca", False)):
    modello = LogisticRegression(max_iter=2000).fit(colonne(A_imp, usa), A_imp.diabete)
    righe.append((nome, roc_auc_score(A_ver.diabete, modello.predict_proba(colonne(A_ver, usa))[:, 1]),
                  roc_auc_score(B.diabete, modello.predict_proba(colonne(B, usa))[:, 1])))
pd.DataFrame(righe, columns=["il modello", "AUROC nell'ospedale A", "AUROC nell'ospedale B"]).round(3)

In A il dato mancante vuol dire *non era sospetto*, e aiuta. In B lo stesso dato mancante vuol dire *ha meno di 50 anni*,
e il modello che ci si era appoggiato perde. Il modello che stima il valore mancante rende meno in A, ma regge in B.

---

### Cosa abbiamo visto

1. **Tenere solo chi ha il dato** regge solo se il dato manca per caso (MCAR).
2. **Stimare il dato mancante** regge se il motivo della mancanza è registrato e la stima lo usa (MAR).
3. **Usare la mancanza come predittore** regge finché il processo che crea i dati mancanti resta lo stesso.
4. **Se il dato manca proprio a chi sta peggio** (MNAR) nessun approccio basta: servono un'analisi di sensibilità,
   il recupero del dato su un campione di pazienti, la registrazione della causa.

### Da chiedere in gara

> **Sotto quale ipotesi sono stati gestiti i dati mancanti? Se il modello usa la mancanza, in quali ospedali è stato verificato?**